In [1]:
import os 
%matplotlib widget
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import scipy.io as sio
import h5py
import datetime
from scipy.integrate import simps
from tqdm import tqdm
from sleep_analysis_functions import compute_spo2_clean

%load_ext autoreload
%autoreload 2

In [2]:
def get_grass_start_end_time(starttime_raw, endtime_raw):
    
    time_str_elements = starttime_raw.flatten()
    start_time = ''.join(chr(time_str_elements[j]) for j in range(time_str_elements.shape[0]))
    time_str_elements = endtime_raw.flatten()
    end_time = ''.join(chr(time_str_elements[j]) for j in range(time_str_elements.shape[0]))

    start_time = start_time.split(':')
    second_elements = start_time[-1].split('.')
    start_time = datetime.datetime(1990,1,1,hour=int(float(start_time[0])), minute=int(float(start_time[1])),
        second=int(float(second_elements[0])), microsecond=int(float('0.'+second_elements[1])*1000000))
    end_time = end_time.split(':')
    second_elements = end_time[-1].split('.')
    end_time = datetime.datetime(1990,1,1,hour=int(float(end_time[0])), minute=int(float(end_time[1])),
        second=int(float(second_elements[0])), microsecond=int(float('0.'+second_elements[1])*1000000))

    return start_time, end_time

def check_load_mgh_dataset(data_path, label_path, channels=None, report_and_actual_time_tol=300, reverse_sign=False):

    gender = None
    handedness = None
    
    try: # usually Grass, saved as pre 7.3 format:
        ff = sio.loadmat(signalfilepath)
        data_path = os.path.basename(signalfilepath)
        if 's' not in ff:
            raise Exception('No signal found in %s.'%data_path)
        signal = ff['s']
        if reverse_sign:
            signal = -signal
        channel_names = [ff['hdr'][0,i]['signal_labels'][0].upper().replace('EKG','ECG') for i in range(ff['hdr'].shape[1])]
        gender = None
        handedness = None
        if 'grass' in signalfilepath:
            Fs = 200.
        else:
            raise Exception('Safety check to make sure this is Grass data with Fs=200')
        # Label part for grass:
        if 'grass' in signalfilepath:
            # load labels
            with h5py.File(labelfilepath) as ffl:
                sleep_stage = ffl['stage'][()].flatten()
                apnea = ffl['apnea'][()].flatten()
                start_time, end_time = get_grass_start_end_time(ffl['features']['StartTime'][()].flatten(), ffl['features']['EndTime'][()].flatten())


    except: # saved as .mat 7.3. new grass or natus files:
        with h5py.File(signalfilepath,'r') as ff:

            hdr = ff['hdr']
            signal_labels = hdr['signal_labels'][:]
            channel_names = [''.join(list(map(chr, ff[signal_labels[i,0]][:]))) for i in range(signal_labels.shape[0])]
            channel_names = [channel.upper() for channel in channel_names]
            signal = ff['s'][()]
            signal = np.transpose(signal);

            if 'recording' in ff.keys(): # only available for natus:
                year = int(np.squeeze(ff['recording']['year']))
                month = int(np.squeeze(ff['recording']['month']))
                day = int(np.squeeze(ff['recording']['day']))
                hour = int(np.squeeze(ff['recording']['hour']))
                if (hour >= 7) and (hour <=10):         # 'typo' by sleep techs
                    hour = hour + 12
                minute = int(np.squeeze(ff['recording']['minute']))
                second = int(np.squeeze(ff['recording']['second']))
                millisecond = int(np.squeeze(ff['recording']['millisecond']))
                Fs = int(np.squeeze(ff['recording']['samplingrate']))

                start_time = datetime.datetime(1990,1,1,hour=hour, minute=minute,
                        second=second, microsecond=int(millisecond*1000))
                end_time = start_time+datetime.timedelta(seconds=max(signal.shape)/Fs)

            else: # grass:
                if 'grass' in signalfilepath:
                    Fs = 200.
                else:
                    raise Exception('Safety check to make sure this is Grass data with Fs=200')

            # Label part for grass:
            ff.close()

        # load labels
        with h5py.File(labelfilepath) as ffl:
            sleep_stage = ffl['stage'][()].flatten()
            apnea = ffl['apnea'][()].flatten()
            if 'grass' in signalfilepath:
                start_time, end_time = get_grass_start_end_time(ffl['features']['StartTime'][()].flatten(), ffl['features']['EndTime'][()].flatten())
            ffl.close()

    # end of loading part
    ##################################
    
    # check signal length = sleep stage length
    if sleep_stage.shape[0]!=signal.shape[1]:
        raise Exception('Inconsistent sleep stage length (%d) and signal length (%d) in %s'%(sleep_stage.shape[0],signal.shape[1],data_path))

    # only take signal channels to study
    if channels is None:
        signal_channel_ids = list(range(len(channel_names)))
        
    elif 'SumEffort' in channels:
        signal_channel_ids = []
        signal_names = []
        for ichannel in ['ABD','CHEST']:
            #channel_name_pattern = re.compile(channels[i][:2].upper()+'-*'+channels[i][-2:].upper())
            found = False
            for j in range(len(channel_names)):
                if channel_names[j]==ichannel.upper():
                    signal_channel_ids.append(j)
                    signal_names.append(channel_names[j])
                    found = True
                    break
            if not found:
                raise Exception('Channel %s is not found.'%ichannel)
        signal = signal[signal_channel_ids,:]#.T
        # do effort belt average here:
        signal = np.sum(signal,0,keepdims=1)/2
        
    else:
        signal_channel_ids = []
        signal_names = []
        for i in range(len(channels)):
            #channel_name_pattern = re.compile(channels[i][:2].upper()+'-*'+channels[i][-2:].upper())
            found = False
            for j in range(len(channel_names)):
                if channel_names[j]==channels[i].upper():
                    signal_channel_ids.append(j)
                    signal_names.append(channel_names[j])
                    found = True
                    break
            if not found:
                continue
#                 raise Exception('Channel %s is not found.'%channels[i])
        signal = signal[signal_channel_ids,:]#.T

    # check whether the signal contains NaN
    if np.any(np.isnan(signal)):
        raise Exception('Found Nan in signal in %s'%data_path)

    # check whether sleep_stage contains all 5 stages
    stages = np.unique(sleep_stage[np.logical_not(np.isnan(sleep_stage))]).astype(int).tolist()
    if len(stages)<=1:
        if len(stages)==0:
            raise Exception('no sleep stages available in %s'%data_path)
        if stages[0] != 5: # only reasonable 1 sleepstage if all is Wake (==5), else raise Error
            raise Exception('#sleep stage <= 1: %s in %s'%(stages,data_path))

    params = {'Fs':Fs*1.0, 'signal_channel_ids':signal_channel_ids, 'signal_names':signal_names}
    if gender is not None:
        params['gender'] = gender
    if handedness is not None:
        params['handedness'] = handedness
    return signal, sleep_stage, apnea, params



In [3]:
signalfilepath = '/media/mad3/Datasets_ConvertedData/sleeplab/grass_data/Abbott_BonnieC_080608_2217.000/Signal_Abbott_BonnieC_080608_2217.000.mat'
labelfilepath = '/media/mad3/Datasets_ConvertedData/sleeplab/grass_data/Abbott_BonnieC_080608_2217.000/Labels_Abbott_BonnieC_080608_2217.000.mat'

In [4]:
signal, sleep_stage, apnea, params = check_load_mgh_dataset(signalfilepath, labelfilepath, channels=['SAO2','SPO2'], report_and_actual_time_tol=300, reverse_sign=False)
signal_names = params['signal_names']
fs = params['Fs']

TypeError: only integer scalar arrays can be converted to a scalar index

In [5]:
assert signal.shape[0] == 1
spo2_signal = np.squeeze(signal)

In [7]:
pd.Series(spo2_signal).to_csv('sample_spo2.csv', index=False)
pd.Series(sleep_stage).to_csv('sample_stage.csv', index=False)

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:1: FutureWarning: The signature of `Series.to_csv` was aligned to that of `DataFrame.to_csv`, and argument 'header' will change its default value from False to True: please pass an explicit value to suppress this warning.
  """Entry point for launching an IPython kernel.
/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:2: FutureWarning: The signature of `Series.to_csv` was aligned to that of `DataFrame.to_csv`, and argument 'header' will change its default value from False to True: please pass an explicit value to suppress this warning.
  


In [77]:
spo2_signal = pd.read_csv('sample_spo2.csv')
sleep_stage = pd.read_csv('sample_stage.csv')

spo2_signal = np.squeeze(np.array(spo2_signal))
sleep_stage = np.squeeze(np.array(sleep_stage))

data = pd.DataFrame()
data['spo2'] = spo2_signal

spo2_signal_clean = compute_spo2_clean(data, fs=200)
spo2_signal_clean = np.squeeze(np.array(spo2_signal_clean['spo2_clean']))

In [91]:
from sleep_analysis_functions import desaturation_detection, calculate_resaturation_periods, hypoxaemic_burden_minutes

In [89]:
hypoxaemic_burden, T90desaturation, T90nonspecific, start_points, end_points = hypoxaemic_burden_minutes(spo2_signal_clean, fs, verbose=True, return_startendpoints=True)

total burden (min): 7.712666666666667
T90 desaturation (min): 7.712666666666667
T90 nonspecific (min): 0.0


In [90]:
# desaturation[desaturation == 0] = np.nan

plt.figure()

plt.plot(sleep_stage + 100, 'k')

plt.plot(spo2_signal, 'k')
plt.plot(spo2_signal_clean,'b')
plt.scatter(start_points, spo2_signal_clean[start_points], c='orange', s=6)
plt.scatter(end_points, spo2_signal_clean[end_points], c='r', s=6)

plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …